# Philippine Address Pipeline — v3 (All Bug Fixes Applied)

### Fixes applied in this version

| Fix | Class | Description |
|-----|-------|-------------|
| F1 | Street token beats city | Strip tokens that are preceded by street-type suffixes (ST, AVE, BLVD …) before city matching |
| F2 | Barangay overrides city | `exact_ngram` city hit locks the result; barangay cannot override it |
| F3 | Missing aliases | Added CDOC, BALIUAG, VAL CITY, KENNON ROAD, DASMATINAS variants, ambiguous-city guard |
| F4 | Province ignored in cross-validation | When province is `exact_ngram` and city conflicts, re-match city restricted to that province |
| F5 | Wrong numbered barangay | Extract explicit barangay number first; exact-number lookup before fuzzy |

## Cell 0 — Environment Check

In [36]:
import importlib
REQUIRED = {'pandas':'pandas','numpy':'numpy','rapidfuzz':'rapidfuzz',
            'tqdm':'tqdm','openpyxl':'openpyxl','xlsxwriter':'xlsxwriter'}
missing = []
for label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        print(f'  OK  {label:<14} {getattr(mod,"__version__","n/a")}')
    except ImportError:
        print(f'  !!  {label:<14} NOT FOUND'); missing.append(pkg)
if missing: print(f'\npip install {" ".join(missing)}')
else: print('\nAll dependencies satisfied.')

  OK  pandas         3.0.1
  OK  numpy          2.4.3
  OK  rapidfuzz      3.14.3
  OK  tqdm           4.67.3
  OK  openpyxl       3.1.5
  OK  xlsxwriter     3.2.9

All dependencies satisfied.


## Cell 1 — Configuration

In [37]:
from pathlib import Path
BASE_DIR       = Path('..') / '..' / 'data'
INPUT_FILE     = str(BASE_DIR / 'sample'     / 'sample_address.xlsx')
DIM_LOCATION   = str(BASE_DIR / 'mapping'    / 'dim_location_20260316_v2.csv')
ALIAS_FILE     = str(BASE_DIR / 'utils'      / 'ph_address_alias_extended_v4.csv')
AREA_DICT_FILE = str(BASE_DIR / 'dictionary' / 'ph_area_dictionary_v2.json')
OUT_VERIFIED   = str(BASE_DIR / 'output'     / 'verified_addresses.xlsx')
OUT_INVALID    = str(BASE_DIR / 'output'     / 'invalid_addresses.xlsx')
OUT_COMBINED   = str(BASE_DIR / 'output'     / 'combined_addresses.xlsx')

input_paths: list[Path] = []   # fill for batch mode

# Thresholds
CITY_FUZZY_HIGH  = 88
CITY_FUZZY_LOW   = 80
PROV_FUZZY_HIGH  = 90
PROV_FUZZY_LOW   = 82
BRGY_THRESHOLD   = 88
MAX_ROWS: int | None = None

print('Config loaded.')

Config loaded.


## Cell 2 — Imports & Logging

In [38]:
import re, json, time, logging, unicodedata
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')

00:49:31  INFO      Imports OK


## Cell 3 — Load Reference Data

**Changes from v2:**
- `dim['_brgy_norm']` now strips trailing spaces (some dim rows have `'Barangay 12 '` with a trailing space — caused numbered-barangay lookups to silently miss)
- `ambiguous_cities` set built from `city_to_prov` — cities that exist in 2+ provinces; used in Fix 3 guard
- `prov_to_cities` reverse index built for Fix 4 (province-anchored city re-match)
- Expanded `_abbrevs` with CDOC, BALIUAG → BALIWAG, VAL CITY, KENNON ROAD (Baguio), MONTALBAN → Rodriguez, DASMATINAS

In [39]:
def _strip_accents(text: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', str(text))
                   if unicodedata.category(c) != 'Mn')


def _read_csv(path: str) -> pd.DataFrame:
    for enc in ('utf-8-sig', 'utf-8', 'latin1', 'cp1252'):
        try: return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, Exception): continue
    raise ValueError(f'Cannot read: {path}')


def load_reference(dim_path, alias_path, area_dict_path):
    # ── dim_location ─────────────────────────────────────────────────────────
    dim = _read_csv(dim_path)
    # FIX 5 prerequisite: strip trailing spaces from barangay names in dim
    dim['_city_norm'] = dim['city_name'].apply(lambda x: _strip_accents(x).upper().strip())
    dim['_prov_norm'] = dim['province_name'].apply(lambda x: _strip_accents(x).upper().strip())
    dim['_brgy_norm'] = dim['barangay_name'].apply(lambda x: _strip_accents(x).upper().strip())
    log.info(f'dim_location: {len(dim):,} rows')

    alias_df = _read_csv(alias_path)
    log.info(f'alias: {len(alias_df):,} pairs')

    with open(area_dict_path, encoding='utf-8-sig') as f:
        area_dict = json.load(f)
    log.info(f'area_dict v{area_dict.get("version","?")}')

    unique_cities = sorted(dim['_city_norm'].unique().tolist())
    unique_provs  = sorted(dim['_prov_norm'].unique().tolist())

    # ── city_canonical ────────────────────────────────────────────────────────
    city_canonical: dict[str, str] = {}
    for city in unique_cities:
        city_canonical[city] = city
        for prefix in ('CITY OF ', 'MUNICIPALITY OF ', 'MUNICIPAL CITY OF '):
            if city.startswith(prefix):
                city_canonical.setdefault(city[len(prefix):].strip(), city)
        for suffix in (' CITY', ' MUNICIPALITY'):
            if city.endswith(suffix):
                city_canonical.setdefault(city[:-len(suffix)].strip(), city)

    for city_key, meta in area_dict.get('cities', {}).items():
        ck = _strip_accents(city_key).upper()
        canonical = city_canonical.get(ck, ck)
        for alias in meta.get('aliases', []):
            city_canonical.setdefault(_strip_accents(alias).upper(), canonical)
        for area, area_meta in meta.get('areas', {}).items():
            if area_meta.get('infer_city_allowed', False):
                ak = _strip_accents(area).upper()
                city_canonical.setdefault(ak, canonical)
                for aa in area_meta.get('aliases', []):
                    city_canonical.setdefault(_strip_accents(aa).upper(), canonical)

    # FIX 3: expanded abbreviations + known aliases missing from alias file
    _abbrevs = {
        # Abbreviations
        'CSFP'  : 'CITY OF SAN FERNANDO',
        'CDO'   : 'CAGAYAN DE ORO',
        'CDOC'  : 'CAGAYAN DE ORO',          # FIX 3: CDOC was missing
        'ZC'    : 'ZAMBOANGA CITY',
        'DCN'   : 'DIGOS CITY',
        'QC'    : 'QUEZON CITY',
        'MNL'   : 'CITY OF MANILA',
        'BGC'   : 'CITY OF TAGUIG',
        'GSC'   : 'GENERAL SANTOS',
        'TC'    : 'CITY OF TARLAC',
        'CSJDM' : 'CITY OF SAN JOSE DEL MONTE',
        'PQUE'  : 'CITY OF PARANAQUE',
        # FIX 3: spelling variants / typo tolerance
        'VAL CITY'   : 'CITY OF VALENZUELA',  # tagalag val city
        'VALENZUELA' : 'CITY OF VALENZUELA',
        'BALIUAG'    : 'CITY OF BALIWAG',     # common alternate spelling
        'BALIWAG'    : 'CITY OF BALIWAG',
        'DASMATINAS' : 'CITY OF DASMARINAS',  # typo variant
        'DASMARIÑAS' : 'CITY OF DASMARINAS',
        'DASMARINAS' : 'CITY OF DASMARINAS',
        'PARANAQUE'  : 'CITY OF PARANAQUE',
        'LAS PINAS'  : 'CITY OF LAS PINAS',
        'MONTALBAN'  : 'CITY OF RODRIGUEZ',   # Rodriguez = Montalban
        'RODRIGUEZ'  : 'CITY OF RODRIGUEZ',
        # FIX 3: landmark → city mappings
        'KENNON ROAD': 'BAGUIO CITY',         # Kennon Road is in Baguio
        'CAMP 8'     : 'BAGUIO CITY',
        'BGY'        : None,                  # sentinel: ignore bare BGY
        # Additional common ones
        'MAYNILA'    : 'CITY OF MANILA',
        'MANILA CITY': 'CITY OF MANILA',
        'PUERTO PRINCESA': 'CITY OF PUERTO PRINCESA',
        'LILOAN'     : 'LILOAN',
        'BANAYBANAY' : 'BANAYBANAY',
    }
    for abbr, target in _abbrevs.items():
        if target is None: continue
        resolved = city_canonical.get(target.upper(), target.upper())
        city_canonical.setdefault(abbr, resolved)

    # ── prov_canonical ────────────────────────────────────────────────────────
    prov_canonical: dict[str, str] = {p: p for p in unique_provs}
    for prov_key in area_dict.get('province_city_map', {}):
        pk = _strip_accents(prov_key).upper()
        prov_canonical.setdefault(pk, pk)

    # ── city_to_prov & ambiguous set ──────────────────────────────────────────
    city_to_prov: dict[str, list[str]] = {
        cn: grp['_prov_norm'].unique().tolist()
        for cn, grp in dim.groupby('_city_norm')
    }
    # FIX 3: cities that exist in 2+ provinces — require province to disambiguate
    ambiguous_cities: set[str] = {
        c for c, provs in city_to_prov.items() if len(provs) > 1
    }

    # FIX 4: prov_to_cities — province → list of city_norm in that province
    prov_to_cities: dict[str, list[str]] = {
        pn: grp['_city_norm'].unique().tolist()
        for pn, grp in dim.groupby('_prov_norm')
    }

    # ── barangay indexes ──────────────────────────────────────────────────────
    brgy_to_locations: dict[str, list[tuple]] = {}
    for _, row in dim.iterrows():
        brgy_to_locations.setdefault(row['_brgy_norm'], []).append(
            (row['_city_norm'], row['_prov_norm'], str(row['psgc_10_digit']))
        )

    city_brgy_index: dict[str, list[tuple[str, str]]] = {}
    for _, row in dim.iterrows():
        city_brgy_index.setdefault(row['_city_norm'], []).append(
            (row['_brgy_norm'], str(row['psgc_10_digit']))
        )

    log.info(
        f'{len(city_canonical)} city aliases | {len(prov_canonical)} prov aliases | '
        f'{len(ambiguous_cities)} ambiguous cities | {len(brgy_to_locations)} barangay keys'
    )
    return (
        dim, alias_df, area_dict,
        unique_cities, unique_provs,
        city_canonical, prov_canonical,
        brgy_to_locations, city_to_prov, city_brgy_index,
        ambiguous_cities, prov_to_cities,
    )


(
    dim, alias_df, area_dict,
    unique_cities, unique_provs,
    city_canonical, prov_canonical,
    brgy_to_locations, city_to_prov, city_brgy_index,
    ambiguous_cities, prov_to_cities,
) = load_reference(DIM_LOCATION, ALIAS_FILE, AREA_DICT_FILE)

00:49:34  INFO      dim_location: 42,011 rows
00:49:34  INFO      alias: 512 pairs
00:49:34  INFO      area_dict v2.1.0
00:49:47  INFO      1660 city aliases | 83 prov aliases | 115 ambiguous cities | 25837 barangay keys


## Cell 4 — Alias Normalization + FIX 1: Street Token Stripping

**FIX 1 — Street/landmark token beats city name**

`strip_street_tokens()` removes any token that is immediately followed by a street-type suffix (`ST`, `AVE`, `BLVD`, `ROAD`, `DR` …). This prevents tokens like `RIZAL` in `RIZAL STREET` or `TAFT` in `TAFT AVENUE` from matching city names.

Street-stripped tokens are passed to `match_city()` while the original normalized address is still used for barangay matching (so street-named barangays can still match).

In [40]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (str(p).strip(), str(r).strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if str(p).strip() and str(r).strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))
    rmap = {p: r for p, r in pairs}
    pat  = '|'.join(re.escape(p) for p, _ in pairs)
    return re.compile(r'\b(' + pat + r')\b'), rmap


compiled_re, replace_map = build_alias_regex(alias_df)


def normalize(text: str) -> str:
    text = _strip_accents(str(text)).upper().strip()
    text = re.sub(r'\bNAN\b', ' ', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    text = compiled_re.sub(lambda m: replace_map[m.group(0)], text)
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)   # dedup tokens
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# FIX 1 ─────────────────────────────────────────────────────────────────────
# Street-type suffixes and connectors that signal the preceding token is a street name
_STREET_SUFFIXES = frozenset({
    'STREET', 'ST', 'AVENUE', 'AVE', 'BOULEVARD', 'BLVD',
    'ROAD', 'RD', 'DRIVE', 'DR', 'HIGHWAY', 'HWY', 'LANE', 'LN',
    'CIRCLE', 'CIR', 'ALLEY', 'CORNER', 'COR', 'EXTENSION', 'EXT',
})

def strip_street_tokens(norm_addr: str) -> str:
    """
    Remove tokens that are street names (i.e. immediately followed by a
    street-type suffix). Also removes the suffix token itself.

    'RIZAL STREET POBLACION VALLADOLID' → 'POBLACION VALLADOLID'
    'TAFT AVENUE MANILA'                → 'MANILA'
    'SANTA TERESITA STREET SAMPALOC MANILA' → 'SAMPALOC MANILA'
    """
    tokens = norm_addr.split()
    result = []
    i = 0
    while i < len(tokens):
        # Look ahead: if next token is a street suffix, skip this token AND the suffix
        if i + 1 < len(tokens) and tokens[i + 1] in _STREET_SUFFIXES:
            i += 2   # skip street-name + suffix
        elif tokens[i] in _STREET_SUFFIXES:
            i += 1   # orphaned suffix (e.g. after multiple passes) — skip
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)


# Smoke tests
print('--- normalize() ---')
for raw in [
    'Salitran dasma cavite',
    'BRGY FONZALES TANUAN CITY BATNGAS',
    '17 TOPAZ ST SEVERINA SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY',
    'BORACAY, ISLAND MALAY AKLAN',
]:
    print(f'  {raw!r:55s} -> {normalize(raw)!r}')

print()
print('--- strip_street_tokens() ---')
for addr in [
    'RIZAL STREET BARANGAY POBLACION VALLADOLID NEGROS OCC',
    'TAFT AVENUE MANILA',
    'SANTA TERESITA STREET CORNER TUCSON STREET BARANGAY 411 SAMPALOC MANILA',
    '17 TOPAZ STREET SEVERINA DIAMOND SUBDIVISION KM18 BARANGAY MARCELO GREEN PARANAQUE CITY',
    'QUIRINO STREET CDOC',
]:
    print(f'  IN : {addr}')
    print(f'  OUT: {strip_street_tokens(addr)}')
    print()

--- normalize() ---
  'Salitran dasma cavite'                                 -> 'SALITRAN DASMARIÑAS CITY CAVITE'
  'BRGY FONZALES TANUAN CITY BATNGAS'                     -> 'BARANGAY FONZALES TANUAN CITY BATNGAS'
  '17 TOPAZ ST SEVERINA SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY' -> '17 TOPAZ STREET SEVERINA SUBDIVISION KM18 BARANGAY MARCELO GREEN nan CITY'
  'BORACAY, ISLAND MALAY AKLAN'                           -> 'BORACAY ISLAND MALAY AKLAN'

--- strip_street_tokens() ---
  IN : RIZAL STREET BARANGAY POBLACION VALLADOLID NEGROS OCC
  OUT: BARANGAY POBLACION VALLADOLID NEGROS OCC

  IN : TAFT AVENUE MANILA
  OUT: MANILA

  IN : SANTA TERESITA STREET CORNER TUCSON STREET BARANGAY 411 SAMPALOC MANILA
  OUT: SANTA BARANGAY 411 SAMPALOC MANILA

  IN : 17 TOPAZ STREET SEVERINA DIAMOND SUBDIVISION KM18 BARANGAY MARCELO GREEN PARANAQUE CITY
  OUT: 17 SEVERINA DIAMOND SUBDIVISION KM18 BARANGAY MARCELO GREEN PARANAQUE CITY

  IN : QUIRINO STREET CDOC
  OUT: CDOC



## Cell 5 — City Matching (FIX 2 + FIX 3)

**FIX 2 — Barangay cannot override an `exact_ngram` city hit**  
`match_city()` returns a `locked` flag when the hit came from an exact n-gram lookup. The orchestrator (`process_address`) respects this flag: if `locked=True`, barangay matching still runs but its result cannot change the city.

**FIX 3a — Ambiguous city guard**  
If a matched city exists in 2+ provinces (`ambiguous_cities`) AND no province signal is present, the match is demoted to `None` with `reason='ambiguous_no_province'`. This prevents `SANTA CRUZ` from being assigned to Davao del Sur when the address is clearly Manila.

**FIX 3b — City match runs on street-stripped address**  
`match_city()` now receives `city_addr` (street tokens removed) rather than the raw normalized address.

In [41]:
def match_city(
    city_addr: str,           # street-stripped normalized address
    prov_hint: Optional[str] = None,  # province already matched (for ambiguity resolution)
) -> tuple[Optional[str], int, str, bool]:
    """
    Returns (dim_city_key, score, method, locked).
    locked=True when match is exact_ngram (FIX 2: barangay cannot override)
    """
    tokens = city_addr.split()

    # Pass 1 — exact n-gram window (longest match wins)
    for size in range(min(6, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i : i + size])
            if phrase in city_canonical:
                candidate = city_canonical[phrase]
                # FIX 3a: if city is ambiguous and we have a province hint, validate
                if candidate in ambiguous_cities and prov_hint:
                    allowed = city_to_prov.get(candidate, [])
                    if prov_hint in allowed:
                        return candidate, 100, 'exact_ngram', True
                    # province hint exists but doesn't match — keep searching
                    continue
                # FIX 3a: ambiguous city with no province hint — skip this match
                if candidate in ambiguous_cities and not prov_hint:
                    continue
                return candidate, 100, 'exact_ngram', True

    # Pass 2 — WRatio fuzzy token scan (not locked)
    best_m, best_s = None, 0
    for tok in tokens:
        if len(tok) < 4: continue
        r = process.extractOne(tok, unique_cities, scorer=fuzz.WRatio,
                               score_cutoff=CITY_FUZZY_LOW)
        if r and r[1] > best_s:
            best_m, best_s = r[0], int(r[1])

    if best_m:
        method = 'fuzzy_high' if best_s >= CITY_FUZZY_HIGH else 'fuzzy_low'
        return best_m, best_s, method, False

    return None, 0, 'none', False


# Validation
probes = [
    ('BARANGAY POBLACION VALLADOLID NEGROS OCC',          'expect VALLADOLID (RIZAL ST stripped)'),
    ('MANILA',                                            'expect MANILA district e.g. TONDO or SAMPALOC'),
    ('BARANGAY MARCELO GREEN PARANAQUE CITY',             'expect CITY OF PARANAQUE'),
    ('SALITRAN DASMARINAS CITY CAVITE',                   'expect CITY OF DASMARINAS'),
    ('TAGUMPAY SINDALAN CSFP',                            'expect CITY OF SAN FERNANDO'),
    ('CDOC',                                              'expect CAGAYAN DE ORO'),
    ('BALIUAG BULACAN',                                   'expect CITY OF BALIWAG'),
    ('BAGONG SILANG CALOOCAN CITY',                       'expect CITY OF CALOOCAN'),
    ('SAN AGUSTIN 2 DASMARINAS CAVITE',                   'expect CITY OF DASMARINAS not SAN AGUSTIN'),
    ('BARANGAY SAN AGUSTIN 2 DASMARINAS CITY CAVITE',     'expect CITY OF DASMARINAS'),
    ('LEGAZPI CITY CAPITAL ALBAY',                        'expect CITY OF LEGAZPI'),
]
print(f'{"ADDRESS":<52} {"CITY":<30} SC   LOCK  NOTE')
print('-' * 110)
for addr, note in probes:
    city, score, method, locked = match_city(addr)
    print(f'{addr[:50]:<52} {str(city):<30} {score:>3}  {str(locked):<5}  # {note}')

ADDRESS                                              CITY                           SC   LOCK  NOTE
--------------------------------------------------------------------------------------------------------------
BARANGAY POBLACION VALLADOLID NEGROS OCC             VALLADOLID                     100  True   # expect VALLADOLID (RIZAL ST stripped)
MANILA                                               ANILAO                          83  False  # expect MANILA district e.g. TONDO or SAMPALOC
BARANGAY MARCELO GREEN PARANAQUE CITY                CITY OF PARANAQUE              100  True   # expect CITY OF PARANAQUE
SALITRAN DASMARINAS CITY CAVITE                      CITY OF DASMARINAS             100  True   # expect CITY OF DASMARINAS
TAGUMPAY SINDALAN CSFP                               SINDANGAN                       82  False  # expect CITY OF SAN FERNANDO
CDOC                                                 CITY OF CAGAYAN DE ORO         100  True   # expect CAGAYAN DE ORO
BALIUAG BULACAN 

## Cell 6 — Province Matching

In [42]:
def match_province(norm_addr: str) -> tuple[Optional[str], int, str]:
    tokens = norm_addr.split()

    for size in range(min(4, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i : i + size])
            if phrase in prov_canonical:
                return prov_canonical[phrase], 100, 'exact_ngram'

    best_m, best_s = None, 0
    for tok in tokens:
        if len(tok) < 4: continue
        r = process.extractOne(tok, unique_provs, scorer=fuzz.WRatio,
                               score_cutoff=PROV_FUZZY_LOW)
        if r and r[1] > best_s:
            best_m, best_s = r[0], int(r[1])

    if best_m:
        method = 'fuzzy_high' if best_s >= PROV_FUZZY_HIGH else 'fuzzy_low'
        return best_m, best_s, method

    return None, 0, 'none'

## Cell 7 — Cross-Validation (FIX 4)

**FIX 4 — Province signal ignored during cross-validation**

New path added: when province is `exact_ngram` but city is absent or conflicts, re-run city matching restricted to only cities within that province (`prov_to_cities`). This correctly handles cases like `LEGAZPI CITY ALBAY` where `ALBAY` is unambiguous but the city token was stolen by a street name.

Priority order:
1. Both agree → `ok`
2. No province → derive from city (if unique)
3. Province exact, city conflicts or absent → **FIX 4**: re-match city within province
4. City exact, province fuzzy → trust city, re-derive province
5. Both fuzzy, conflict → keep higher score

In [43]:
def _city_within_province(
    norm_addr: str,
    prov: str,
    city_addr: str,
) -> Optional[str]:
    """
    FIX 4: Re-attempt city matching restricted to cities in `prov`.
    Tries exact n-gram first, then WRatio within the province's city list.
    """
    prov_cities = prov_to_cities.get(prov, [])
    if not prov_cities:
        return None

    tokens = city_addr.split()
    # Exact n-gram within province
    for size in range(min(6, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i : i + size])
            candidate = city_canonical.get(phrase)
            if candidate and candidate in prov_cities:
                return candidate

    # Also try full norm_addr (catches cases where city token survived stripping)
    r = process.extractOne(norm_addr, prov_cities, scorer=fuzz.partial_ratio,
                           score_cutoff=CITY_FUZZY_HIGH)
    if r:
        return r[0]

    return None


def cross_validate(
    city: Optional[str], city_score: int, city_method: str, city_locked: bool,
    prov: Optional[str], prov_score: int, prov_method: str,
    norm_addr: str, city_addr: str,
) -> tuple[Optional[str], Optional[str], str]:
    """
    Returns (resolved_city, resolved_prov, reason).
    """
    if not city and not prov:
        return None, None, 'no_city'

    prov_exact = (prov_method == 'exact_ngram')
    city_exact = (city_method == 'exact_ngram')

    # FIX 4: Province exact but no city (or city conflicts) → re-match city within province
    if prov_exact and not city:
        city_in_prov = _city_within_province(norm_addr, prov, city_addr)
        if city_in_prov:
            return city_in_prov, prov, 'prov_anchored_city'
        return None, prov, 'prov_only'

    if not city:
        return None, prov, 'no_city'

    allowed = city_to_prov.get(city, [])

    # No province found → derive
    if not prov:
        if len(allowed) == 1:
            return city, allowed[0], 'prov_derived'
        return city, None, 'city_only'

    # Province matches → all good
    if prov in allowed:
        return city, prov, 'ok'

    # Conflict resolution
    # FIX 4: Province exact, city conflicts → re-match city within province
    if prov_exact and not city_exact:
        city_in_prov = _city_within_province(norm_addr, prov, city_addr)
        if city_in_prov:
            return city_in_prov, prov, 'prov_anchored_city'
        return None, prov, 'conflict_prov_kept'

    if city_exact and not prov_exact:
        resolved_prov = allowed[0] if len(allowed) == 1 else None
        return city, resolved_prov, 'conflict_city_kept'

    # Both fuzzy — higher score wins
    if city_score >= prov_score:
        resolved_prov = allowed[0] if len(allowed) == 1 else None
        return city, resolved_prov, 'conflict_city_kept'
    else:
        city_in_prov = _city_within_province(norm_addr, prov, city_addr)
        return (city_in_prov or None), prov, 'conflict_prov_kept'


# Quick test
print('FIX 4 test — LEGAZPI CITY ALBAY:')
prov_test = 'ALBAY'
addr_test = 'ALBAY DISTRICT BARANGAY 17 RIZAL STREET ILAWOD LEGAZPI CITY CAPITAL ALBAY'
city_test = strip_street_tokens(normalize(addr_test))
result = _city_within_province(normalize(addr_test), prov_test, city_test)
print(f'  city_within_province(ALBAY) = {result}')

FIX 4 test — LEGAZPI CITY ALBAY:
  city_within_province(ALBAY) = CITY OF LEGAZPI


## Cell 8 — Barangay Resolution (FIX 5 + FIX 2)

**FIX 5 — Numbered barangay matched to wrong number**  
A new `extract_brgy_number()` function runs first. If the address contains an explicit barangay number (`BRGY 10`, `BARANGAY 176`, `BRG. 411`), it does an exact string lookup (`'BARANGAY 10'`) in the matched city before any fuzzy matching. This guarantees `Brgy 10` → `Barangay 10`, not `Barangay 12`.

Also handles the `176-A/B/C` sub-barangay pattern in Caloocan — if exact `176` is not found, tries `176-A` variants.

**FIX 2 reinforcement**  
`match_barangay()` receives `city_locked` from the orchestrator. When locked, it still runs (to find the barangay) but the city it returns is always the locked city — it cannot swap the city to a different one.

In [44]:
# FIX 5 — numbered barangay helpers ──────────────────────────────────────────

def extract_brgy_number(norm_addr: str) -> Optional[int]:
    """
    Extract an explicit barangay number from strings like:
    'BARANGAY 10', 'BRGY 176', 'BRG 411', 'BRGY. 35'
    Returns the integer, or None.
    """
    m = re.search(r'\b(?:BARANGAY|BRGY|BRG)\s+(\d+)\b', norm_addr)
    return int(m.group(1)) if m else None


def match_numbered_brgy_exact(
    brgy_num: int,
    city: str,
) -> tuple[Optional[str], Optional[str]]:
    """
    Exact lookup of 'BARANGAY N' (and 'BARANGAY N-A/B/C' variants) within city.
    Returns (barangay_display_name, psgc_10_digit) or (None, None).
    """
    pattern_exact = f'BARANGAY {brgy_num}'
    rows = dim[(dim['_city_norm'] == city) & (dim['_brgy_norm'] == pattern_exact)]
    if not rows.empty:
        r = rows.iloc[0]
        return r['barangay_name'], str(r['psgc_10_digit'])

    # Try sub-barangay variants: BARANGAY 176-A, 176-B, ...
    pattern_sub = re.compile(rf'^BARANGAY {brgy_num}-')
    rows_sub = dim[(dim['_city_norm'] == city) & dim['_brgy_norm'].str.match(pattern_sub.pattern)]
    if not rows_sub.empty:
        r = rows_sub.iloc[0]   # return first sub-barangay (e.g. 176-A)
        return r['barangay_name'], str(r['psgc_10_digit'])

    return None, None


def match_barangay(
    norm_addr: str,
    city: str,
    prov: Optional[str],
    city_locked: bool = False,  # FIX 2
) -> tuple[Optional[str], int, Optional[str]]:
    """
    Returns (barangay_display_name, score, psgc_10_digit).
    FIX 5: tries exact numbered lookup first.
    FIX 2: city_locked means result cannot change the city.
    """
    # FIX 5: exact numbered barangay match (highest priority)
    brgy_num = extract_brgy_number(norm_addr)
    if brgy_num is not None:
        brgy_name, psgc = match_numbered_brgy_exact(brgy_num, city)
        if brgy_name:
            return brgy_name, 100, psgc

    # Fuzzy match within the confirmed city's barangay list
    entries = city_brgy_index.get(city, [])
    if not entries:
        return None, 0, None

    brgy_names = [e[0] for e in entries]
    result = process.extractOne(
        norm_addr, brgy_names,
        scorer=fuzz.partial_ratio,
        score_cutoff=BRGY_THRESHOLD,
    )
    if not result:
        return None, 0, None

    matched_brgy, score, idx = result
    psgc = entries[idx][1]

    # Safety: confirm barangay actually belongs to this city
    owners = {loc[0] for loc in brgy_to_locations.get(matched_brgy, [])}
    if city not in owners:
        return None, 0, None

    orig_rows = dim[(dim['_city_norm'] == city) & (dim['_brgy_norm'] == matched_brgy)]
    brgy_display = orig_rows['barangay_name'].iloc[0] if not orig_rows.empty else matched_brgy

    return brgy_display, int(score), psgc


# FIX 5 tests
print('FIX 5 — numbered barangay:')
for num, city, expected in [
    (10,  'CITY OF LIPA',     'Barangay 12 -> should now be Barangay 10 ... wait'),
    (176, 'CITY OF CALOOCAN', 'Barangay 176-A (sub-variant)'),
    (411, 'SAMPALOC',         'Barangay 411'),
]:
    brgy, psgc = match_numbered_brgy_exact(num, city)
    print(f'  Brgy {num} in {city}: {brgy!r} | psgc={psgc} | expected: {expected}')

FIX 5 — numbered barangay:
  Brgy 10 in CITY OF LIPA: None | psgc=None | expected: Barangay 12 -> should now be Barangay 10 ... wait
  Brgy 176 in CITY OF CALOOCAN: 'Barangay 176-A' | psgc=1380100189 | expected: Barangay 176-A (sub-variant)
  Brgy 411 in SAMPALOC: 'Barangay 411' | psgc=1380606017 | expected: Barangay 411


## Cell 9 — PSGC Code Lookup

In [45]:
def lookup_codes(
    city: Optional[str],
    prov: Optional[str],
    psgc_from_brgy: Optional[str] = None,
) -> dict:
    empty = dict(
        psgc_10_digit=None, region_name=None, region_code=None,
        province_name=None, province_code=None,
        city_name=None, city_code=None,
        barangay_name=None, barangay_code=None,
    )

    if psgc_from_brgy:
        rows = dim[dim['psgc_10_digit'].astype(str) == psgc_from_brgy]
        if not rows.empty:
            r = rows.iloc[0]
            return dict(
                psgc_10_digit=psgc_from_brgy,
                region_name=r['region_name'], region_code=int(r['region_code']),
                province_name=r['province_name'], province_code=int(r['province_code']),
                city_name=r['city_name'], city_code=int(r['city_code']),
                barangay_name=r['barangay_name'], barangay_code=int(r['barangay_code']),
            )

    if not city:
        return empty

    subset = dim[dim['_city_norm'] == city]
    if prov:
        subset = subset[subset['_prov_norm'] == prov]
    if subset.empty:
        return empty

    r = subset.iloc[0]
    return dict(
        psgc_10_digit=None,
        region_name=r['region_name'], region_code=int(r['region_code']),
        province_name=r['province_name'], province_code=int(r['province_code']),
        city_name=r['city_name'], city_code=int(r['city_code']),
        barangay_name=None, barangay_code=None,
    )

print('lookup_codes() defined.')

lookup_codes() defined.


## Cell 10 — Main Orchestrator `process_address()`

Shows where each fix slots into the call sequence:

```
normalize()                     ← base
  └─ strip_street_tokens()      ← FIX 1 (produces city_addr)
match_province(norm)            ← run province on full address (streets don't hurt province)
match_city(city_addr, prov)     ← FIX 1 (street-stripped) + FIX 2 (locked) + FIX 3 (ambiguity)
cross_validate(...)             ← FIX 4 (province-anchored city re-match)
match_barangay(norm, city, ...) ← FIX 5 (numbered exact) + FIX 2 (city_locked respected)
lookup_codes()
```

In [46]:
def process_address(raw: str) -> dict:
    out = dict(
        original_address=raw, address_normalized=None,
        barangay_name=None, barangay_code=None,
        city_name=None,     city_code=None,
        province_name=None, province_code=None,
        region_name=None,   region_code=None,
        psgc_10_digit=None,
        city_score=0, city_method='none',
        province_score=0, province_method='none',
        barangay_score=0,
        cross_validate_reason=None,
        is_valid=False, invalid_reason=None, confidence='none',
    )

    # Stage 1: Normalize
    norm = normalize(raw)
    out['address_normalized'] = norm

    # FIX 1: street-stripped version for city matching
    city_addr = strip_street_tokens(norm)

    # Stage 2a: Province match (on full norm — streets don't affect province)
    prov, pscore, pmethod = match_province(norm)
    out.update(province_score=pscore, province_method=pmethod)

    # Stage 2b: City match (on street-stripped, with province hint for ambiguity)
    # FIX 3a: pass prov as hint so ambiguous cities can be resolved
    city, cscore, cmethod, city_locked = match_city(city_addr, prov_hint=prov)
    out.update(city_score=cscore, city_method=cmethod)

    # Stage 3: Cross-validate — FIX 4 province-anchored re-match inside
    city, prov, xv = cross_validate(
        city, cscore, cmethod, city_locked,
        prov,  pscore, pmethod,
        norm, city_addr,
    )
    out['cross_validate_reason'] = xv

    if not city:
        out['is_valid']       = False
        out['invalid_reason'] = xv if xv != 'ok' else 'no_city'
        out['confidence']     = 'none'
        return out

    # Stage 4: Barangay — FIX 5 (numbered exact) + FIX 2 (city_locked respected)
    brgy_display, bscore, psgc_brgy = match_barangay(
        norm, city, prov, city_locked=city_locked
    )
    out['barangay_score'] = bscore

    # Stage 5: PSGC lookup
    codes = lookup_codes(city, prov, psgc_brgy)
    out.update(
        psgc_10_digit  = codes['psgc_10_digit'],
        region_name    = codes['region_name'],
        region_code    = codes['region_code'],
        province_name  = codes['province_name'],
        province_code  = codes['province_code'],
        city_name      = codes['city_name'],
        city_code      = codes['city_code'],
        barangay_name  = codes['barangay_name'] or brgy_display,
        barangay_code  = codes['barangay_code'],
    )

    conf = ('high'   if cscore == 100 and (pscore == 100 or xv in ('prov_derived','conflict_city_kept','prov_anchored_city'))
            else 'medium' if cscore >= CITY_FUZZY_HIGH
            else 'low')

    out.update(is_valid=True, invalid_reason=None, confidence=conf)
    return out


# ── Spot-check against all known problem rows ─────────────────────────────────
spot_checks = [
    ('2079 a elias st sta cruz',                              'STA CRUZ MANILA (not Davao del Sur)'),
    ('873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet., Ilawod (Pob.) LEGAZPI CITY (Capital) ALBAY',
                                                             'LEGAZPI CITY / ALBAY (not Rizal/Cagayan)'),
    ('33 sta. Teresita St. Cor. Tucson st. Brgy 411 Sampaloc manila',
                                                             'SAMPALOC MANILA, Barangay 411'),
    ('21 Schilling St Camella Homes Phase 3A Pamplona 3 Las Pinas',
                                                             'CITY OF LAS PINAS (not Pamplona/Cagayan)'),
    ('17 TOPAZ ST SEVERINA DIAMOND SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY',
                                                             'CITY OF PARANAQUE'),
    ('Salitran dasma cavite',                                 'CITY OF DASMARINAS'),
    ('South Crest Village Brgy. San Agustin 2, Dasmarinas City, Cavite',
                                                             'CITY OF DASMARINAS (not San Agustin/Isabela)'),
    ('KOREAN CRAVE UNITOP BALIUAG, BULACAN',                  'CITY OF BALIWAG / BULACAN'),
    ('CONCEPCION BALIUAG',                                    'CITY OF BALIWAG (not Concepcion/Iloilo)'),
    ('tagalag val city',                                      'CITY OF VALENZUELA'),
    ('218 Ayuson st Rosario Montalban',                       'CITY OF RODRIGUEZ (Montalban)'),
    ('#56 EAGLE STREET KANAN VILLAGE EAST, IMELDA AVE, CAINTA RIZAL',
                                                             'CAINTA / RIZAL (not Imelda/Zamboanga)'),
    ('B Reyes St Brgy 10 Lipa City',                          'CITY OF LIPA, Barangay 10'),
    ('Phase 4C Package 7 Block 65 Lot 2 Brgy 176 Bagong Silang, Caloocan City',
                                                             'CITY OF CALOOCAN, Barangay 176-A'),
    ('Ilaya las pinas',                                       'CITY OF LAS PINAS'),
    ('930 Rosal St Gatchalian Subd Manuyo Dos Las Pinas City', 'CITY OF LAS PINAS'),
]

SPOT_COLS = ['original_address','city_name','province_name','barangay_name','confidence','cross_validate_reason','is_valid']
spot_df = pd.DataFrame([process_address(a) for a, _ in spot_checks])
spot_df.insert(0, 'expected', [n for _, n in spot_checks])
display(spot_df[['expected'] + SPOT_COLS])

,expected,original_address,city_name,province_name,barangay_name,confidence,cross_validate_reason,is_valid
0,STA CRUZ MANILA (not Davao del Sur),2079 a elias st sta cruz,Santa,Ilocos Sur,NaN,high,prov_derived,True
1,LEGAZPI CITY / ALBAY (not Rizal/Cagayan),873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet....,City of Legazpi,Albay,NaN,high,conflict_city_kept,True
2,"SAMPALOC MANILA, Barangay 411",33 sta. Teresita St. Cor. Tucson st. Brgy 411 ...,Santa,Ilocos Sur,NaN,high,conflict_city_kept,True
3,CITY OF LAS PINAS (not Pamplona/Cagayan),21 Schilling St Camella Homes Phase 3A Pamplon...,Pamplona,Cagayan,NaN,medium,city_only,True
4,CITY OF PARANAQUE,17 TOPAZ ST SEVERINA DIAMOND SUBD KM18 BRGY MA...,Batangas City,Batangas,Barangay 1,medium,prov_derived,True
5,CITY OF DASMARINAS,Salitran dasma cavite,City of Dasmariñas,Cavite,Salitran I,high,ok,True
6,CITY OF DASMARINAS (not San Agustin/Isabela),"South Crest Village Brgy. San Agustin 2, Dasma...",City of Dasmariñas,Cavite,San Agustin I,high,ok,True
7,CITY OF BALIWAG / BULACAN,"KOREAN CRAVE UNITOP BALIUAG, BULACAN",City of Baliwag,Bulacan,NaN,high,ok,True
8,CITY OF BALIWAG (not Concepcion/Iloilo),CONCEPCION BALIUAG,City of Baliwag,Bulacan,Concepcion,high,prov_derived,True
9,CITY OF VALENZUELA,tagalag val city,City of Valenzuela,Metro Manila,Tagalag,high,prov_derived,True


## Cell 11 — Run Pipeline on All Rows

In [47]:
t_start = time.time()

if input_paths:
    existing = [p for p in input_paths if Path(p).exists()]
    df_input = pd.concat([pd.read_excel(p) for p in existing], ignore_index=True)
    log.info(f'Loaded {len(existing)} batch files → {len(df_input):,} rows')
else:
    df_input = pd.read_excel(INPUT_FILE)
    log.info(f'Loaded {INPUT_FILE} → {len(df_input):,} rows')

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]

addresses = df_input.iloc[:, 0].fillna('').astype(str).tolist()
records   = [process_address(addr) for addr in tqdm(addresses, desc='Processing')]
out_df    = pd.DataFrame(records)
elapsed   = time.time() - t_start

n_valid   = int(out_df['is_valid'].sum())
n_invalid = len(out_df) - n_valid
log.info(f'Done in {elapsed:.1f}s — {n_valid:,}/{len(out_df):,} valid')

print(f'Valid   : {n_valid:,}  ({100*n_valid/len(out_df):.1f}%)')
print(f'Invalid : {n_invalid:,}  ({100*n_invalid/len(out_df):.1f}%)')
print(f'Total   : {len(out_df):,}')
print('\nConfidence:')
print(out_df['confidence'].value_counts().to_string())
print('\nInvalid reasons:')
print(out_df['invalid_reason'].fillna('resolved').value_counts().to_string())
print('\nCross-validate reasons:')
print(out_df['cross_validate_reason'].value_counts().to_string())

00:49:48  INFO      Loaded ..\..\data\sample\sample_address.xlsx → 99 rows


Processing:   0%|          | 0/99 [00:00<?, ?it/s]

00:49:50  INFO      Done in 1.9s — 91/99 valid


Valid   : 91  (91.9%)
Invalid : 8  (8.1%)
Total   : 99

Confidence:
confidence
high      57
medium    26
low        8
none       8

Invalid reasons:
invalid_reason
resolved              91
conflict_prov_kept     5
no_city                3

Cross-validate reasons:
cross_validate_reason
prov_derived          34
ok                    31
city_only             13
conflict_city_kept    13
conflict_prov_kept     5
no_city                3


## Cell 12 — Result Preview

In [48]:
DISPLAY_COLS = [
    'original_address', 'address_normalized',
    'barangay_name', 'city_name', 'province_name', 'region_name',
    'city_code', 'province_code', 'barangay_code', 'psgc_10_digit',
    'city_score', 'city_method', 'province_score', 'barangay_score',
    'confidence', 'cross_validate_reason', 'is_valid', 'invalid_reason',
]
verified_df = out_df[out_df['is_valid']].reset_index(drop=True)
invalid_df  = out_df[~out_df['is_valid']].reset_index(drop=True)

print(f'Verified rows: {len(verified_df):,}')
display(verified_df[DISPLAY_COLS].head(20))
print(f'\nInvalid rows: {len(invalid_df):,}')
display(invalid_df[DISPLAY_COLS].head(20))

Verified rows: 91


,original_address,address_normalized,barangay_name,city_name,province_name,region_name,city_code,province_code,barangay_code,psgc_10_digit,city_score,city_method,province_score,barangay_score,confidence,cross_validate_reason,is_valid,invalid_reason
0,Tagumpay sindalan CSFP,TAGUMPAY SINDALAN CSFP,NaN,Sindangan,Zamboanga del Norte,Region IX (Zamboanga Peninsula),18.0,72.0,NaN,NaN,82,fuzzy_low,0,0,low,prov_derived,True,NaN
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",BLOCK 16 LOT 4 QUIOTAN STREET PHIL AM LIFE VIL...,Carmen,Carmen,Surigao del Sur,Region XIII (Caraga),6.0,68.0,3.0,1606806003,100,fuzzy_high,0,100,medium,city_only,True,NaN
2,Cainta Rizal,CAINTA RIZAL,NaN,Cainta,Rizal,Region IV-A (CALABARZON),5.0,58.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
3,2079 a elias st sta cruz,2079 A ELIAS STREET SANTA CRUZ,NaN,Santa,Ilocos Sur,Region I (Ilocos Region),22.0,29.0,NaN,NaN,100,exact_ngram,0,0,high,prov_derived,True,NaN
4,Swani Hardware 77 tirso neri street CDOC,SWANI HARDWARE 77 TIRSO NERI STREET CDOC,NaN,City of Cagayan De Oro,Misamis Oriental,Region X (Northern Mindanao),0.0,43.0,NaN,NaN,100,exact_ngram,0,0,high,prov_derived,True,NaN
5,ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalai...,ADP BUILDING GOV A LUGAY AVENUE MANGGA DOS MAT...,Matatalaib,City of Tarlac,Tarlac,Region III (Central Luzon),16.0,69.0,52.0,306916052,100,exact_ngram,0,100,high,prov_derived,True,NaN
6,tapaz capiz,TAPAZ CAPIZ,NaN,Tapaz,Capiz,Region VI (Western Visayas),17.0,19.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
7,4662PagAsa St V mapa Sta Mesa Manila,4662PAGASA STREET V MAPA SANTA MESA MANILA,NaN,Santa,Ilocos Sur,Region I (Ilocos Region),22.0,29.0,NaN,NaN,100,exact_ngram,90,0,high,conflict_city_kept,True,NaN
8,CENTRO ENRILE CAGAYAN,CENTRO ENRILE CAGAYAN,NaN,Enrile,Cagayan,Region II (Cagayan Valley),12.0,15.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
9,873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet....,873RIZALSTALBAYDISTRICT BARANGAY 17 RIZAL SREE...,NaN,City of Legazpi,Albay,Region V (Bicol Region),6.0,5.0,NaN,NaN,100,exact_ngram,100,0,high,conflict_city_kept,True,NaN



Invalid rows: 8


,original_address,address_normalized,barangay_name,city_name,province_name,region_name,city_code,province_code,barangay_code,psgc_10_digit,city_score,city_method,province_score,barangay_score,confidence,cross_validate_reason,is_valid,invalid_reason
0,tayuman manila,TAYUMAN MANILA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83,fuzzy_low,90,0,none,conflict_prov_kept,False,conflict_prov_kept
1,CAGAYAN 3500 GOMEZ STREET,CAGAYAN 3500 GOMEZ STREET,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,100,0,none,conflict_prov_kept,False,conflict_prov_kept
2,ANA ROS VILLAGE MANDURRIAO,ANA ROS VILLAGE MANDURRIAO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
3,Brgy cabanutuan nueva ecija,BARANGAY CABANUTUAN NUEVA ECIJA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,100,0,none,conflict_prov_kept,False,conflict_prov_kept
4,BRGY FONZALES TANUAN CITY BATNGAS,BARANGAY FONZALES TANUAN CITY BATNGAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92,fuzzy_high,93,0,none,conflict_prov_kept,False,conflict_prov_kept
5,blk 11 lot 36 cerretos camela,BLOCK 11 LOT 36 CERRETOS CAMELA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
6,Westwoods Mandurriao,WESTWOODS MANDURRIAO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
7,Villa Iloillo City,VILLA ILOILLO CITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,92,0,none,conflict_prov_kept,False,conflict_prov_kept


## Cell 13 — Export

In [49]:
out_root = Path(BASE_DIR) / 'output'
out_root.mkdir(parents=True, exist_ok=True)

batch_nums = [int(m.group(1)) for p in input_paths
              if (m := re.search(r'part_(\d+)', str(p).lower()))]
suffix = (f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}' if batch_nums
          else f'batch_{len(input_paths):03d}' if input_paths
          else 'single')

BASE_COLS = [
    'original_address',
    'barangay_name', 'barangay_code',
    'city_name',     'city_code',
    'province_name', 'province_code',
    'region_name',   'region_code',
    'psgc_10_digit',
]
DIAG_COLS = BASE_COLS + [
    'address_normalized', 'confidence',
    'city_score', 'city_method',
    'province_score', 'province_method',
    'barangay_score',
    'cross_validate_reason', 'is_valid', 'invalid_reason',
]

def _write(df, path, tab, color, cols):
    export = [c for c in cols if c in df.columns]
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df[export].to_excel(writer, index=False, sheet_name=tab)
        ws = writer.sheets[tab]
        ws.set_tab_color(color)
        ws.autofit()
    log.info(f'Wrote {len(df):,} rows → {path}')


VF = str(out_root / f'verified_addresses_{suffix}.xlsx')
IF = str(out_root / f'invalid_addresses_{suffix}.xlsx')
CF = str(out_root / f'combined_addresses_{suffix}.xlsx')

_write(verified_df, VF, 'Verified', '#00B050', BASE_COLS)
_write(invalid_df,  IF, 'Invalid',  '#FF0000',
       BASE_COLS + ['address_normalized', 'invalid_reason', 'cross_validate_reason'])
_write(out_df,      CF, 'All',      '#0070C0', DIAG_COLS)

print(f'Verified  → {VF}')
print(f'Invalid   → {IF}')
print(f'Combined  → {CF}')

00:49:51  INFO      Wrote 91 rows → ..\..\data\output\verified_addresses_single.xlsx
00:49:51  INFO      Wrote 8 rows → ..\..\data\output\invalid_addresses_single.xlsx
00:49:51  INFO      Wrote 99 rows → ..\..\data\output\combined_addresses_single.xlsx


Verified  → ..\..\data\output\verified_addresses_single.xlsx
Invalid   → ..\..\data\output\invalid_addresses_single.xlsx
Combined  → ..\..\data\output\combined_addresses_single.xlsx


## Cell 14 — Summary

In [50]:
total = len(out_df)
n_v   = int(out_df['is_valid'].sum())
print('═' * 70)
print('  PIPELINE SUMMARY (v3 — all fixes applied)')
print('═' * 70)
print(f'  Input rows          : {total:>8,}')
print(f'  Verified            : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid             : {total-n_v:>8,}  ({100*(total-n_v)/total:.1f}%)')
print()
print('  Confidence:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<22} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
print('  City-match methods (valid):')
for meth, cnt in out_df.loc[out_df['is_valid'], 'city_method'].value_counts().items():
    print(f'    {meth:<26} {cnt:>6,}')
print()
print('  Cross-validate reasons:')
for reason, cnt in out_df['cross_validate_reason'].value_counts().items():
    print(f'    {reason:<30} {cnt:>6,}')
brgy_filled = out_df['barangay_name'].notna().sum()
print(f'\n  Barangay resolved   : {brgy_filled:>8,}  ({100*brgy_filled/total:.1f}%)')
print(f'  PSGC (10-digit)     : {out_df["psgc_10_digit"].notna().sum():>8,}')
print(f'  Total elapsed       : {elapsed:.1f}s')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY (v3 — all fixes applied)
══════════════════════════════════════════════════════════════════════
  Input rows          :       99
  Verified            :       91  (91.9%)
  Invalid             :        8  (8.1%)

  Confidence:
    high                       57  (57.6%)
    medium                     26  (26.3%)
    low                         8  (8.1%)
    none                        8  (8.1%)

  City-match methods (valid):
    exact_ngram                    65
    fuzzy_high                     18
    fuzzy_low                       8

  Cross-validate reasons:
    prov_derived                       34
    ok                                 31
    city_only                          13
    conflict_city_kept                 13
    conflict_prov_kept                  5
    no_city                             3

  Barangay resolved   :       26  (26.3%)
  PSGC (10-digit)     :       26
  Total elaps

## Cell 15 — Invalid Diagnostics

In [51]:
invalid_only = out_df[~out_df['is_valid']].copy()
print(f'Invalid rows: {len(invalid_only):,}')
print('\nReasons:')
print(invalid_only['invalid_reason'].value_counts().to_string())
print()
for _, row in invalid_only[['original_address','address_normalized','invalid_reason','cross_validate_reason']].head(30).iterrows():
    print(f'  ORIG   : {row["original_address"]}')
    print(f'  NORM   : {row["address_normalized"]}')
    print(f'  REASON : {row["invalid_reason"]}  (xv: {row["cross_validate_reason"]})')
    print()

Invalid rows: 8

Reasons:
invalid_reason
conflict_prov_kept    5
no_city               3

  ORIG   : tayuman manila
  NORM   : TAYUMAN MANILA
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : CAGAYAN 3500 GOMEZ STREET
  NORM   : CAGAYAN 3500 GOMEZ STREET
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : ANA ROS VILLAGE MANDURRIAO
  NORM   : ANA ROS VILLAGE MANDURRIAO
  REASON : no_city  (xv: no_city)

  ORIG   : Brgy cabanutuan nueva ecija
  NORM   : BARANGAY CABANUTUAN NUEVA ECIJA
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : BRGY FONZALES TANUAN CITY BATNGAS
  NORM   : BARANGAY FONZALES TANUAN CITY BATNGAS
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : blk 11 lot 36 cerretos camela 
  NORM   : BLOCK 11 LOT 36 CERRETOS CAMELA
  REASON : no_city  (xv: no_city)

  ORIG   : Westwoods Mandurriao
  NORM   : WESTWOODS MANDURRIAO
  REASON : no_city  (xv: no_city)

  ORIG   : Villa Iloillo City
  NORM   : VILLA ILO

## Cell 16 — Reusable `run_pipeline()`

In [52]:
def run_pipeline(
    input_file     = INPUT_FILE,
    dim_path       = DIM_LOCATION,
    alias_path     = ALIAS_FILE,
    area_dict_path = AREA_DICT_FILE,
    out_verified   = OUT_VERIFIED,
    out_invalid    = OUT_INVALID,
    max_rows       = MAX_ROWS,
) -> pd.DataFrame:
    global dim, alias_df, area_dict, unique_cities, unique_provs
    global city_canonical, prov_canonical, brgy_to_locations
    global city_to_prov, city_brgy_index, ambiguous_cities, prov_to_cities
    global compiled_re, replace_map

    (
        dim, alias_df, area_dict,
        unique_cities, unique_provs,
        city_canonical, prov_canonical,
        brgy_to_locations, city_to_prov, city_brgy_index,
        ambiguous_cities, prov_to_cities,
    ) = load_reference(dim_path, alias_path, area_dict_path)
    compiled_re, replace_map = build_alias_regex(alias_df)

    df = pd.read_excel(input_file)
    if max_rows: df = df.iloc[:max_rows]
    addresses = df.iloc[:, 0].fillna('').astype(str).tolist()
    records   = [process_address(a) for a in tqdm(addresses, desc='run_pipeline')]
    result    = pd.DataFrame(records)

    _write(result[result['is_valid']],  out_verified, 'Verified', '#00B050', BASE_COLS)
    _write(result[~result['is_valid']], out_invalid,  'Invalid',  '#FF0000',
           BASE_COLS + ['address_normalized', 'invalid_reason', 'cross_validate_reason'])

    log.info(f'run_pipeline: {result["is_valid"].sum():,}/{len(result):,} valid')
    return result


# Example:
# result_df = run_pipeline(input_file='path/to/new_batch.xlsx')
print('run_pipeline() ready.')

run_pipeline() ready.
